# Kiểm tra, ép kiểu và chốt dataset đơn hàng đã join

Task #28 (Story #4): đọc kết quả trung gian của Task #27 (`data/processed/orders_step2_enriched.csv`), ép kiểu dữ liệu đúng, kiểm tra sanity check, rồi lưu thành dataset cuối cùng của Story #4.

In [1]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
df = pd.read_csv(PROCESSED_DIR / "orders_step2_enriched.csv", low_memory=False)

print(f"Đọc {len(df)} dòng, {len(df.columns)} cột từ orders_step2_enriched.csv")

Đọc 99441 dòng, 41 cột từ orders_step2_enriched.csv


## Ép kiểu dữ liệu tường minh

Theo phát hiện ở Task #23 (dtype): 5 cột thời gian của `orders` đọc bằng `read_csv` mặc định ra kiểu chuỗi, phải `pd.to_datetime` tường minh.

Theo ghi chú ở Task #27: các cột `payment_has_*` khi lưu CSV rồi đọc lại có 1 dòng thiếu (đơn không có payment) làm dtype thành `object` lẫn `True`/`False`/`NaN` — ép về kiểu boolean có hỗ trợ giá trị thiếu (`"boolean"`).

In [2]:
datetime_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in datetime_cols:
    df[col] = pd.to_datetime(df[col])

bool_cols = [c for c in df.columns if c.startswith("payment_has_")] + ["items_multi_seller"]
for col in bool_cols:
    df[col] = df[col].astype("boolean")

print("Kiểu dữ liệu sau khi ép:")
print(df[datetime_cols + bool_cols].dtypes.to_string())

Kiểu dữ liệu sau khi ép:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
payment_has_boleto                      boolean
payment_has_credit_card                 boolean
payment_has_debit_card                  boolean
payment_has_not_defined                 boolean
payment_has_voucher                     boolean
items_multi_seller                      boolean


## Sanity check: tổng thanh toán so với tổng giá + phí vận chuyển

Nếu join đúng, `payment_total_value` (Task #26) phải xấp xỉ `items_total_price + items_total_freight` (Task #26) với đa số đơn — chênh lệch nhỏ có thể do làm tròn hoặc trả góp, chênh lệch lớn cần xem lại.

In [3]:
both = df.dropna(subset=["payment_total_value", "items_total_price", "items_total_freight"])
diff = (both["payment_total_value"] - (both["items_total_price"] + both["items_total_freight"])).abs()

print(f"So sánh trên {len(both)} đơn có cả payment và items:")
print(f"  Chênh lệch trung bình: {diff.mean():.4f}")
print(f"  % đơn chênh lệch < 0.05: {(diff < 0.05).mean():.2%}")
print(f"  % đơn chênh lệch < 1.00: {(diff < 1.00).mean():.2%}")
print(f"  Chênh lệch lớn nhất: {diff.max():.2f}")
print("\n=> Sai số nhỏ, phần lớn đơn khớp — xác nhận join ở Task #26 không có lỗi cấu trúc.")

So sánh trên 98665 đơn có cả payment và items:
  Chênh lệch trung bình: 0.0332
  % đơn chênh lệch < 0.05: 99.74%
  % đơn chênh lệch < 1.00: 99.75%
  Chênh lệch lớn nhất: 182.81

=> Sai số nhỏ, phần lớn đơn khớp — xác nhận join ở Task #26 không có lỗi cấu trúc.


## Tỉ lệ giá trị thiếu (null) theo cột

In [4]:
null_pct = (df.isna().sum() / len(df) * 100).round(2)
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
print(f"{len(null_pct)}/{len(df.columns)} cột có giá trị thiếu:")
null_pct

17/41 cột có giá trị thiếu:


order_delivered_customer_date     2.98
order_delivered_carrier_date      1.79
primary_seller_zip_code_prefix    0.78
items_num_items                   0.78
items_num_products                0.78
items_num_sellers                 0.78
items_total_price                 0.78
items_total_freight               0.78
primary_seller_id                 0.78
primary_seller_state              0.78
primary_seller_city               0.78
items_num_categories              0.78
review_score_avg                  0.77
review_score_max                  0.77
review_score_min                  0.77
review_count                      0.77
order_approved_at                 0.16
dtype: float64

## Tóm tắt quyết định aggregation đã áp dụng (Story #4)

Để Story #5/#6 dùng nhất quán, không phải tra lại từng Task:

| Bảng nguồn | Quyết định | Task |
|---|---|---|
| `order_items` | Aggregate theo `order_id`: số lượng sản phẩm/seller khác nhau, tổng `price`, tổng `freight_value` | #26 |
| `order_payments` | Giữ đầy đủ (không chọn 1 phương thức chính): tổng tiền, số dòng/loại, tổng tiền theo từng hình thức thanh toán + cờ nhị phân | #26 |
| `customers` | Join thẳng theo `customer_id` (quan hệ 1-1) | #27 |
| `sellers` | Chọn seller đóng góp `price` cao nhất làm seller chính (`primary_seller_*`) + cờ `items_multi_seller` | #27 |
| `products` + category translation | Không chọn category đại diện, chỉ giữ `items_num_categories` | #27 |
| `order_reviews` | Dedupe dòng trùng y hệt rồi tính `review_score_avg`/`min`/`max`/`review_count`, không chọn 1 review đại diện | #27 |
| `geolocation` | **Chưa join** — cần dedupe riêng (~26% trùng zip), để lại cho Story nào thực sự cần lat/lng | — |

**Lưu ý cho Story #5/#6**: 775 đơn (0,78%) không có dữ liệu items/seller/category, 1 đơn không có payment, 768 đơn không có review — các cột tương ứng sẽ là `NaN`, cần quyết định xử lý rõ khi dùng cho mô hình.

In [5]:
OUT_PATH = PROCESSED_DIR / "orders_joined.csv"
df.to_csv(OUT_PATH, index=False)
print(f"Đã lưu dataset cuối: {len(df)} dòng, {len(df.columns)} cột vào {OUT_PATH}")

Đã lưu dataset cuối: 99441 dòng, 41 cột vào ..\data\processed\orders_joined.csv
